# Fine Tuning de Imagenes con AutoGluon

El fine tuning de una red neuronal consiste en ajustar un modelo preentrenado con un conjunto de datos mas pequeno y especifico para nuestra tarea.

En este notebook vamos a entrenar un clasificador de imagenes (bananas maduras vs normales) usando `AutoGluon`. El flujo es completamente programatico y reproducible desde notebook.

## Paso 1: Instalar dependencias

Instalaremos AutoGluon y librerias auxiliares para preparar los datos, entrenar el modelo y visualizar resultados.

> Si estas en Google Colab, ejecuta esta celda y luego reinicia el entorno si te lo pide.

!pip install -q -U autogluon.multimodal pandas scikit-learn pillow requests matplotlib

## Paso 2: Descargar y preparar el dataset de bananas

Usaremos imagenes de ejemplo del curso y construiremos un `DataFrame` con dos columnas:
- `image`: ruta local de la imagen
- `label`: clase (`normal` o `madura`)

Esto es el formato recomendado por `AutoGluon` para clasificacion de imagenes.

In [ ]:
from pathlib import Path
import requests
import pandas as pd

base_url = "https://github.com/amiune/freecodingtour/raw/main/cursos/espanol/deeplearning/data/bananas/"
output_dir = Path("data/bananas")
output_dir.mkdir(parents=True, exist_ok=True)

records = []
for i in range(1, 6):
    for label in ["normal", "madura"]:
        filename = f"{label}{i}.jpeg"
        local_path = output_dir / filename
        if not local_path.exists():
            content = requests.get(base_url + filename, timeout=20)
            content.raise_for_status()
            local_path.write_bytes(content.content)
        records.append({"image": str(local_path), "label": label})

df = pd.DataFrame(records)
df.head()

## Paso 3: Separar entrenamiento y validacion

Haremos una separacion estratificada para conservar la proporcion de clases en ambos subconjuntos.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

print("Tamano train:", len(train_df))
print("Tamano val:", len(val_df))
print("Distribucion train:\n", train_df["label"].value_counts())
print("Distribucion val:\n", val_df["label"].value_counts())

## Paso 4: Entrenar el modelo con AutoGluon

Entrenaremos un clasificador de imagenes con transferencia de aprendizaje (`presets="best_quality"`).

> Nota: para CPU puede tardar bastante; puedes bajar `time_limit` para pruebas rapidas.

In [ ]:
from autogluon.multimodal import MultiModalPredictor

predictor = MultiModalPredictor(
    label="label",
    problem_type="classification",
    path="autogluon_bananas",
)

predictor.fit(
    train_data=train_df,
    tuning_data=val_df,
    presets="best_quality",
    time_limit=600,
)

## Paso 5: Evaluar el modelo

Calculamos metricas sobre el conjunto de validacion.

In [ ]:
metricas = predictor.evaluate(val_df)
metricas

## Paso 6: Probar inferencia con una imagen nueva

Ahora hacemos una prediccion sobre una imagen que no usamos para entrenar.

In [ ]:
test_url = "https://raw.githubusercontent.com/amiune/freecodingtour/main/cursos/espanol/deeplearning/data/bananas/banana.jpg"
test_path = output_dir / "banana_test.jpg"

if not test_path.exists():
    content = requests.get(test_url, timeout=20)
    content.raise_for_status()
    test_path.write_bytes(content.content)

pred_df = pd.DataFrame({"image": [str(test_path)]})

pred_clase = predictor.predict(pred_df)
pred_proba = predictor.predict_proba(pred_df)

print("Clase predicha:", pred_clase.iloc[0])
pred_proba

## Paso 7: Guardar y recargar el modelo entrenado

Asi puedes reutilizar el modelo sin volver a entrenar.

In [ ]:
save_dir = "autogluon_bananas_export"
predictor.save(save_dir)

predictor_cargado = MultiModalPredictor.load(save_dir)
predictor_cargado.predict(pd.DataFrame({"image": [str(test_path)]}))

## Recomendaciones para mejorar resultados

- Aumentar la cantidad y variedad de imagenes por clase.
- Usar una particion de validacion mas grande (o cross-validation).
- Probar distintos `presets` y aumentar `time_limit`.
- Entrenar en GPU para acelerar y habilitar modelos mas grandes.
- Revisar errores con `predictor.predict_proba(...)` para detectar casos ambiguos.

# Fin: [Volver al contenido del curso](https://www.freecodingtour.com/cursos/espanol/deeplearning/deeplearning.html)